# L38 - Surrogate Modeling for Simulation Optimization

**Learning objectives**
- Fit a simple surrogate model to noisy simulation output.
- Use the surrogate to screen promising `(s, S)` policies.
- Distinguish a quick polynomial surrogate from a full kriging or GP model.
- Identify what additional tooling is needed for a full Gaussian-process workflow.

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from simdes.models import SSInventory

## Step 1: collect a small design of experiments

This starter notebook uses a quadratic surrogate that runs with the current project dependencies. If `scikit-learn` is available, the final cell shows how to extend the notebook to a Gaussian-process or kriging-style model.

In [ ]:
def simulate_mean_cost(s: int, S: int, n_reps: int = 8, sim_time: float = 365.0, seed: int = 2026) -> float:
    model = SSInventory(reorder_point=s, order_up_to=S, sim_time=sim_time, seed=seed)
    df = model.run_replications(n_reps)
    return float(df['avg_total_cost'].mean())

design = [(10, 80), (10, 120), (20, 100), (30, 120), (40, 140), (50, 160)]
design_df = pd.DataFrame(design, columns=['s', 'S'])
design_df['mean_cost'] = [simulate_mean_cost(s, S) for s, S in design]
design_df

In [ ]:
def features(s: float, S: float) -> np.ndarray:
    return np.array([1.0, s, S, s * s, S * S, s * S], dtype=float)

X = np.vstack([features(row.s, row.S) for row in design_df.itertuples()])
y = design_df['mean_cost'].to_numpy()
beta, *_ = np.linalg.lstsq(X, y, rcond=None)

def predict_cost(s: float, S: float) -> float:
    return float(features(s, S) @ beta)

grid = [(s, S) for s in range(10, 55, 5) for S in range(80, 181, 10) if S > s]
pred_df = pd.DataFrame(grid, columns=['s', 'S'])
pred_df['predicted_cost'] = [predict_cost(s, S) for s, S in grid]
pred_df.sort_values('predicted_cost').head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(pred_df['s'], pred_df['S'], c=pred_df['predicted_cost'], cmap='viridis')
ax.scatter(design_df['s'], design_df['S'], color='red', marker='x', s=80, label='simulated design points')
ax.set_xlabel('reorder point s')
ax.set_ylabel('order-up-to level S')
ax.set_title('Quadratic surrogate over the policy grid')
ax.legend()
fig.colorbar(scatter, ax=ax, label='predicted cost')
fig.tight_layout()
plt.show()

In [ ]:
try:
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

    kernel = ConstantKernel(1.0) * RBF(length_scale=[10.0, 20.0]) + WhiteKernel(noise_level=1.0)
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0)
    gp.fit(design_df[['s', 'S']], design_df['mean_cost'])
    pred_df['gp_mean'] = gp.predict(pred_df[['s', 'S']])
    pred_df.sort_values('gp_mean').head()
except ImportError:
    print('Optional extension: install scikit-learn to run the GP / kriging example.')

## Try It Yourself

1. Add more design points near the current best predicted policy.
2. Refit the surrogate and compare the new recommended policy.
3. If `scikit-learn` is available, compare the quadratic surrogate to the GP fit.